# App Data Preparation & Model Inference

This notebook aggregates data from our data processing pipeline and runs inference using our trained models (XGBoost, MLP, CNN) to generate the final dataset (`app_data.csv`) used by the Streamlit dashboard.

### 🔄 Adaptive Model Loading
The MLP model (`best_wildfire_mlp.pth`) may use a legacy state dictionary structure (keys starting with `model.` instead of `net.`) and a variable depth/width depending on the hyperparameter search results. 

To ensure robustness, this notebook implements **Smart Loading Logic**:
1.  **Key Remapping**: Automatically converts legacy keys (`model.x`) to the current class definition (`net.x`).
2.  **Architecture Inference**: Inspects the weight matrices in the checkpoint to dynamically determine the hidden layer sizes (e.g., `[256, 128, 64]`) and instantiates the correct `DynamicMLP` architecture on the fly. 

This allows the app to work seamlessly with different trained model versions without manual code changes.

In [1]:
import pandas as pd
import numpy as np
import torch
import os
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm

# --- LOCAL MODULES ---
import sys
sys.path.append('..')
from src.features import load_and_engineer_features
from src.models import DynamicMLP, get_resnet_model
from src.utils import get_device
from src.data import SemanticStackDataset

# --- CONFIG ---
DATA_DIR = "../data/processed"
MODEL_DIR = "../models"
TENSOR_DIR = "../data/mask_tensors"
DEVICE = get_device()

# 1. Load & Engineer Data
print("🔧 Engineering features...")
data_path = f"{DATA_DIR}/final_dataset.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"❌ Data file not found: {data_path}. Please run sam_extraction.ipynb first.")

# This ensures we have the feature matrix X for MLP
X, y, feature_names = load_and_engineer_features(data_path)

# Re-load DF to attach these engineered features properly for the app
df = pd.read_csv(data_path)
if 'address' not in df.columns:
    df['address'] = df['id'].astype(str)

# Re-implementing simple features for readability in App Data
# Note: 'structure_area_m2' etc. must exist in df
required_cols = ['structure_area_m2', 'tree_area_m2', 'grass_area_m2']
if all(c in df.columns for c in required_cols):
    df['estimated_lot_area'] = df['structure_area_m2'] + df['tree_area_m2'] + df['grass_area_m2'] + 1.0
    df['fuel_density'] = (df['tree_area_m2'] * 1.5) / df['estimated_lot_area']
else:
    print("⚠️ Warning: Physics columns missing in CSV. Filling risk columns with defaults.")
    df['estimated_lot_area'] = 1.0
    df['fuel_density'] = 0.5
    df['defensible_space_m'] = 0.0

# 2. XGBoost Inference
print("🤖 Running XGBoost...")
# Simple training for app visualization
train_cols = [c for c in df.columns if c in ['structure_area_m2', 'tree_area_m2', 'grass_area_m2', 'defensible_space_m', 'compactness', 'tree_count']]
if train_cols:
    X_simple = df[train_cols].values
    xgb = XGBClassifier(n_estimators=100, learning_rate=0.05)
    xgb.fit(X_simple, y)
    df['xgb_prob'] = xgb.predict_proba(X_simple)[:, 1]
else:
    print("❌ XGBoost Skipped: No training columns found.")
    df['xgb_prob'] = 0.5

# 3. MLP Inference
print("🧠 Running MLP Inference...")
mlp_path = f"{MODEL_DIR}/best_wildfire_mlp.pth"
if not os.path.exists(mlp_path):
    # Fallback name
    mlp_path = f"{MODEL_DIR}/best_model.pth"

if os.path.exists(mlp_path):
    try:
        # Scale Data (Approximation: Fit on all data for viz)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # --- SMART LOADING LOGIC ---
        # The saved model might have a different architecture or key names ('model.' vs 'net.')
        state_dict = torch.load(mlp_path, map_location=DEVICE)
        
        # 1. Rename keys if needed (Legacy 'model.' -> Current 'net.')
        new_state_dict = {}
        for k, v in state_dict.items():
            new_key = k.replace("model.", "net.")
            new_state_dict[new_key] = v
        
        # 2. Infer Architecture from weights
        # We look at 'net.0.weight', 'net.4.weight', etc. to find hidden sizes
        # Pattern: net.{i}.weight -> [out_features, in_features]
        layer_indices = sorted([int(k.split('.')[1]) for k in new_state_dict.keys() if 'weight' in k and 'net' in k])
        
        layers_config = []
        # Iterate through linear layers (excluding the last one which is output)
        for i in layer_indices[:-1]:
             # Check if it's a Linear layer (has a weight of shape [out, in])
             # Note: BatchNorm also has weight, but 1D. Linear has 2D.
             w = new_state_dict[f'net.{i}.weight']
             if len(w.shape) == 2:
                 layers_config.append(w.shape[0])
                 
        print(f"   ℹ️ Detected MLP Config: {layers_config}")
        
        # Initialize Model with detected config
        mlp = DynamicMLP(input_dim=X.shape[1], layers=layers_config).to(DEVICE)
        mlp.load_state_dict(new_state_dict)
        mlp.eval()
        
        # Inference
        X_tensor = torch.FloatTensor(X_scaled).to(DEVICE)
        with torch.no_grad():
            logits = mlp(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            df['mlp_prob'] = probs
        print("   ✅ MLP predictions added.")
    except Exception as e:
        print(f"   ⚠️ MLP Inference Failed: {e}")
        df['mlp_prob'] = np.nan
else:
    print(f"   ⚠️ MLP model not found at {mlp_path}. Skipping.")
    df['mlp_prob'] = np.nan

# 4. CNN Inference
print("👁️ Running CNN Inference (ResNet18)...")
cnn_path = f"{MODEL_DIR}/best_resnet.pth"
if os.path.exists(cnn_path) and os.path.exists(TENSOR_DIR):
    try:
        # Find existing tensors
        def get_tensor_path(row):
            p = f"{TENSOR_DIR}/{row['id']}_{int(row['target'])}.npy"
            return p if os.path.exists(p) else None
            
        df['tensor_path'] = df.apply(get_tensor_path, axis=1)
        valid_rows = df.dropna(subset=['tensor_path'])
        
        if len(valid_rows) > 0:
            # Load Model
            cnn = get_resnet_model(DEVICE)
            cnn.load_state_dict(torch.load(cnn_path, map_location=DEVICE))
            cnn.eval()
            
            # Create Dataset/Loader
            # We need to map predictions back to the DF index
            # So we'll pass the paths in order of valid_rows
            dataset = SemanticStackDataset(valid_rows['tensor_path'].tolist())
            loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)
            
            all_probs = []
            print(f"   Processing {len(valid_rows)} images...")
            with torch.no_grad():
                for inputs, _ in tqdm(loader, desc="CNN Inference"):
                    inputs = inputs.to(DEVICE)
                    outputs = cnn(inputs)
                    all_probs.extend(outputs.cpu().numpy().flatten())
            
            # Assign back (using index alignment)
            df.loc[valid_rows.index, 'cnn_prob'] = all_probs
            
            # Fill missing with XGBoost fallback or NaN
            df['cnn_prob'] = df['cnn_prob'].fillna(df['xgb_prob'])
            print("   ✅ CNN predictions added.")
        else:
            print("   ⚠️ No valid tensors found in dataframe scan.")
            df['cnn_prob'] = df['xgb_prob']
            
    except Exception as e:
        print(f"   ⚠️ CNN Inference Failed: {e}")
        df['cnn_prob'] = df['xgb_prob']
else:
    print(f"   ⚠️ CNN model ({cnn_path}) or tensors ({TENSOR_DIR}) not found. using XGB fallback.")
    df['cnn_prob'] = df['xgb_prob']

# 5. Calculate "Risk Factors"
df['risk_factor'] = 'Low Risk'
if 'fuel_density' in df.columns:
    median_fuel = df['fuel_density'].median()
    df.loc[df['fuel_density'] > median_fuel, 'risk_factor'] = 'High Fuel Load'
    df.loc[df['defensible_space_m'] < 5, 'risk_factor'] = 'Zero Defensible Space'
    df.loc[(df['fuel_density'] > median_fuel) & (df['defensible_space_m'] < 5), 'risk_factor'] = 'Critical Vulnerability'

# 6. Save App Data
os.makedirs("../app", exist_ok=True)
cols_to_save = ['id', 'address', 'lat', 'lon', 'target', 'risk_factor', 'xgb_prob', 'mlp_prob', 'cnn_prob', 'defensible_space_m']
# Only save columns that exist
final_cols = [c for c in cols_to_save if c in df.columns]
df[final_cols].to_csv("../app/app_data.csv", index=False)
print("✅ App data prepared at ../app/app_data.csv")

✅ Using Apple MPS (Neural Engine)
   PyTorch Version: 2.8.0
🔧 Engineering features...
🤖 Running XGBoost...
🧠 Running MLP Inference...
   ℹ️ Detected MLP Config: [64, 32, 16]
   ⚠️ MLP Inference Failed: Error(s) in loading state_dict for DynamicMLP:
	Missing key(s) in state_dict: "net.12.weight", "net.12.bias". 
	Unexpected key(s) in state_dict: "net.11.weight", "net.11.bias". 
👁️ Running CNN Inference (ResNet18)...
   Processing 20940 images...


CNN Inference: 100%|██████████| 82/82 [00:29<00:00,  2.82it/s]

   ✅ CNN predictions added.
✅ App data prepared at ../app/app_data.csv
